# 11 - 信号分析与可视化

基于 `stock_technical` 表中已有的技术指标数据，进行信号解读、可视化分析和归因汇总。

**前置条件：** 需先运行 `10_data_pipeline.ipynb` 完成数据采集。

**本 Notebook 包含：**
1. 信号层量化评分（使用 `TechnicalSignalGenerator`）
2. K 线 + 布林带 / MACD / RSI 等多维可视化
3. 估值走势 / 累计收益率对比
4. 综合归因汇总

---

## 1. 初始化：数据库连接 + 数据加载

In [67]:
import pandas as pd
from IPython.display import display, HTML
from datetime import datetime, timedelta

from invest_model.db import get_engine
from invest_model.repositories.base import BaseRepository
from invest_model.repositories.stock_daily_repo import StockDailyRepository
from invest_model.repositories.stock_pool_repo import StockPoolRepository
from invest_model.repositories.technical_repo import TechnicalRepository

engine = get_engine()
base_repo = BaseRepository(engine)
daily_repo = StockDailyRepository(engine)
tech_repo = TechnicalRepository(engine)
pool_repo = StockPoolRepository(engine)

core_codes = pool_repo.get_pool_codes("core")
etf_codes = pool_repo.get_pool_codes("etf")
_pool_all = pool_repo.get_pool()
code_name = dict(zip(_pool_all["code"], _pool_all["name"]))

_end = datetime.now().strftime("%Y%m%d")
_start = (datetime.now() - timedelta(days=365)).strftime("%Y%m%d")

stock_data = {}
for code in core_codes:
    daily = daily_repo.get_daily(code, _start, _end)
    tech = tech_repo.get_technical(code, _start, _end)
    if not daily.empty and not tech.empty:
        merged = daily.merge(tech, on=["code", "trade_date"], how="left",
                             suffixes=("", "_tech"))
        merged["trade_date"] = pd.to_datetime(merged["trade_date"], format="%Y%m%d")
        merged = merged.sort_values("trade_date")
        stock_data[code] = merged

print(f"已加载 {len(stock_data)} 只个股的日线+技术指标数据 ({_start}~{_end})")
print(f"ETF 标的: {len(etf_codes)} 只")

已加载 5 只个股的日线+技术指标数据 (20250502~20260502)
ETF 标的: 1 只


## 2. 信号层量化评分

使用 `TechnicalSignalGenerator` 对每只股票生成 7 类技术信号 + 综合评分。

In [68]:
from invest_model.technical.signal_interpreter import TechnicalSignalGenerator

gen = TechnicalSignalGenerator()
snapshots = {}

for code, df in stock_data.items():
    name = code_name.get(code, code)
    try:
        snapshot = gen.generate_snapshot(code, df)
        snapshots[code] = snapshot
    except Exception as e:
        print(f"⚠️ {name} ({code}): 信号生成失败 - {e}")

# 汇总表
signal_rows = []
for code, snap in snapshots.items():
    name = code_name.get(code, code)
    row = {"名称": name, "代码": code, "综合评分": f"{snap.composite_score:+.3f}"}
    for sig in snap.signals:
        row[sig.name] = f"{sig.score:+.2f}"
    signal_rows.append(row)

df_signals = pd.DataFrame(signal_rows)
print(f"信号评分完成: {len(snapshots)} 只\n")
display(df_signals)

# 详细归因
print("\n信号归因详情:")
print("=" * 60)
for code, snap in snapshots.items():
    name = code_name.get(code, code)
    score_str = f"{snap.composite_score:+.3f}"
    print(f"\n【{name}】综合评分: {score_str}")
    print(f"  {snap.summary()}")
    for sig in snap.signals:
        strength_icon = {"strong": "●", "moderate": "◐", "weak": "○"}[sig.strength.value]
        print(f"    {strength_icon} {sig.label}  (score={sig.score:+.3f})")

信号评分完成: 5 只



,名称,代码,综合评分,macd_trend,rsi_extreme,boll_position,ma_bias,vol_ratio,momentum_20,volatility_20
0,粤桂股份,000833.SZ,+0.028,+1.00,-0.53,-0.75,-0.99,+0.00,+0.89,-0.30
1,比亚迪,002594.SZ,-0.215,-0.32,-0.01,-0.75,-0.16,+0.00,+0.01,+0.00
2,卫星化学,002648.SZ,+0.022,+0.71,-0.05,-0.75,-0.46,+0.00,+0.32,-0.30
3,润泽科技,300442.SZ,-0.252,-0.60,+0.00,-0.70,-0.06,+0.00,+0.18,-0.30
4,潞化科技,600691.SH,-0.045,+0.25,-0.00,-0.75,+0.07,+0.00,+0.07,-0.30



信号归因详情:

【粤桂股份】综合评分: +0.028
  MACD 多头强势，趋势向上；RSI 超买区域 (73.2)，短期或有回调压力；价格接近布林上轨 (100%)，波动加大；MA60 大幅偏离上方 (29.56%)，均值回归概率增大；20日动量强劲 (26.63%)，上升趋势明显；20日年化波动率极高 (65.65%)，风险显著
    ● MACD 多头强势，趋势向上  (score=+1.000)
    ◐ RSI 超买区域 (73.2)，短期或有回调压力  (score=-0.530)
    ● 价格接近布林上轨 (100%)，波动加大  (score=-0.750)
    ● MA60 大幅偏离上方 (29.56%)，均值回归概率增大  (score=-0.985)
    ○ 量比 1.18，成交正常  (score=+0.000)
    ● 20日动量强劲 (26.63%)，上升趋势明显  (score=+0.888)
    ◐ 20日年化波动率极高 (65.65%)，风险显著  (score=-0.300)

【比亚迪】综合评分: -0.215
  MACD 死叉，需关注下行风险；价格接近布林上轨 (100%)，波动加大
    ◐ MACD 死叉，需关注下行风险  (score=-0.316)
    ○ RSI 中性 (52.4)  (score=-0.010)
    ● 价格接近布林上轨 (100%)，波动加大  (score=-0.750)
    ○ MA60 偏离度正常 (4.71%)  (score=-0.157)
    ○ 量比 1.02，成交正常  (score=+0.000)
    ○ 20日动量中性 (0.32%)  (score=+0.011)
    ○ 20日年化波动率正常 (32.78%)  (score=+0.000)

【卫星化学】综合评分: +0.022
  MACD 多头强势，趋势向上；价格接近布林上轨 (100%)，波动加大；MA60 偏离上方 (13.73%)，关注回归风险；20日动量偏多 (9.59%)；20日年化波动率极高 (51.86%)，风险显著
    ● MACD 多头强势，趋势向上  (score=+0.707)
    ○ RSI 中性 (61.3)  (score=-

---

## 3. 可视化分析

以下图表对 stock_pool 中所有个股进行多维度技术面、基本面可视化分析。

In [69]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

### 3.1 K 线 + 布林带

In [70]:
for code, df in stock_data.items():
    name = code_name.get(code, code)
    fig = make_subplots(rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.03,
                        row_heights=[0.7, 0.3],
                        subplot_titles=[f"{name} ({code}) K线 + 布林带", "成交量"])

    fig.add_trace(go.Candlestick(
        x=df["trade_date"], open=df["open"], high=df["high"],
        low=df["low"], close=df["close"], name="K线",
        increasing_line_color="#EF5350", decreasing_line_color="#26A69A",
    ), row=1, col=1)

    for col, lbl, dash in [("boll_upper", "上轨", "dot"),
                            ("boll_mid", "中轨", "solid"),
                            ("boll_lower", "下轨", "dot")]:
        if col in df.columns:
            fig.add_trace(go.Scatter(
                x=df["trade_date"], y=df[col], name=lbl,
                line=dict(width=1, dash=dash), opacity=0.7,
            ), row=1, col=1)

    colors = ["#EF5350" if c >= o else "#26A69A"
              for c, o in zip(df["close"], df["open"])]
    fig.add_trace(go.Bar(
        x=df["trade_date"], y=df["volume"], name="成交量",
        marker_color=colors, opacity=0.6,
    ), row=2, col=1)

    fig.update_layout(height=500, xaxis_rangeslider_visible=False,
                      showlegend=True, template="plotly_white",
                      margin=dict(t=40, b=20))
    fig.show()

### 3.2 MACD

In [71]:
for code, df in stock_data.items():
    name = code_name.get(code, code)
    fig = go.Figure()

    fig.add_trace(go.Scatter(
        x=df["trade_date"], y=df["macd_dif"], name="DIF",
        line=dict(color="#2196F3", width=1.5),
    ))
    fig.add_trace(go.Scatter(
        x=df["trade_date"], y=df["macd_dea"], name="DEA",
        line=dict(color="#FF9800", width=1.5),
    ))

    hist_colors = ["#EF5350" if v >= 0 else "#26A69A" for v in df["macd_hist"].fillna(0)]
    fig.add_trace(go.Bar(
        x=df["trade_date"], y=df["macd_hist"], name="MACD柱",
        marker_color=hist_colors, opacity=0.7,
    ))

    fig.update_layout(title=f"{name} ({code}) MACD", height=300,
                      template="plotly_white", margin=dict(t=40, b=20))
    fig.show()

### 3.3 RSI (6日 / 14日)

In [72]:
for code, df in stock_data.items():
    name = code_name.get(code, code)
    fig = go.Figure()

    fig.add_trace(go.Scatter(
        x=df["trade_date"], y=df["rsi_6"], name="RSI(6)",
        line=dict(color="#AB47BC", width=1.5),
    ))
    fig.add_trace(go.Scatter(
        x=df["trade_date"], y=df["rsi_14"], name="RSI(14)",
        line=dict(color="#42A5F5", width=1.5),
    ))

    fig.add_hline(y=70, line_dash="dash", line_color="red", opacity=0.5,
                  annotation_text="超买 70")
    fig.add_hline(y=30, line_dash="dash", line_color="green", opacity=0.5,
                  annotation_text="超卖 30")
    fig.add_hline(y=50, line_dash="dot", line_color="gray", opacity=0.3)

    fig.update_layout(title=f"{name} ({code}) RSI", height=300,
                      yaxis=dict(range=[0, 100]),
                      template="plotly_white", margin=dict(t=40, b=20))
    fig.show()

### 3.4 60 日线偏离度（多标的对比）

In [73]:
fig = go.Figure()
for code, df in stock_data.items():
    name = code_name.get(code, code)
    fig.add_trace(go.Scatter(
        x=df["trade_date"], y=df["ma60_bias"], name=name,
        line=dict(width=1.5),
    ))

fig.add_hline(y=0, line_dash="solid", line_color="gray", opacity=0.5)
fig.update_layout(title="60 日线偏离度对比 (MA60 Bias)",
                  yaxis_title="偏离度", yaxis_tickformat=".2%",
                  height=400, template="plotly_white", margin=dict(t=40, b=20))
fig.show()

### 3.5 成交量比率 / 动量 / 波动率

In [74]:
for code, df in stock_data.items():
    name = code_name.get(code, code)
    fig = make_subplots(rows=3, cols=1, shared_xaxes=True, vertical_spacing=0.06,
                        subplot_titles=["量比 (Vol Ratio)", "20日动量", "20日年化波动率"])

    fig.add_trace(go.Scatter(
        x=df["trade_date"], y=df["vol_ratio"], name="量比",
        line=dict(color="#7E57C2", width=1.2),
    ), row=1, col=1)
    fig.add_hline(y=1, line_dash="dash", line_color="gray", opacity=0.4, row=1, col=1)

    mom_colors = ["#EF5350" if v >= 0 else "#26A69A" for v in df["momentum_20"].fillna(0)]
    fig.add_trace(go.Bar(
        x=df["trade_date"], y=df["momentum_20"], name="动量",
        marker_color=mom_colors, opacity=0.7,
    ), row=2, col=1)

    fig.add_trace(go.Scatter(
        x=df["trade_date"], y=df["volatility_20"], name="波动率",
        line=dict(color="#FF7043", width=1.2), fill="tozeroy", opacity=0.5,
    ), row=3, col=1)

    fig.update_layout(title=f"{name} ({code}) 量比 / 动量 / 波动率",
                      height=550, showlegend=False,
                      template="plotly_white", margin=dict(t=60, b=20))
    fig.show()

### 3.6 估值走势 (PE / PB)

In [75]:
for code in core_codes:
    name = code_name.get(code, code)
    funda = base_repo.read_sql(
        "SELECT trade_date, pe_ttm, pb FROM stock_fundamental "
        "WHERE code = :code AND trade_date BETWEEN :s AND :e ORDER BY trade_date",
        {"code": code, "s": _start, "e": _end},
    )
    if funda.empty:
        print(f"{name} ({code}): 无估值数据，跳过")
        continue

    funda["trade_date"] = pd.to_datetime(funda["trade_date"], format="%Y%m%d")

    fig = make_subplots(specs=[[{"secondary_y": True}]])
    fig.add_trace(go.Scatter(
        x=funda["trade_date"], y=funda["pe_ttm"], name="PE(TTM)",
        line=dict(color="#1E88E5", width=1.5),
    ), secondary_y=False)
    fig.add_trace(go.Scatter(
        x=funda["trade_date"], y=funda["pb"], name="PB",
        line=dict(color="#F4511E", width=1.5),
    ), secondary_y=True)

    fig.update_layout(title=f"{name} ({code}) 估值走势", height=350,
                      template="plotly_white", margin=dict(t=40, b=20))
    fig.update_yaxes(title_text="PE(TTM)", secondary_y=False)
    fig.update_yaxes(title_text="PB", secondary_y=True)
    fig.show()

### 3.7 多标的累计收益率对比

将所有个股和 ETF 以期初收盘价归一化，对比区间内累计涨跌幅。

In [76]:
fig = go.Figure()

# 个股
for code, df in stock_data.items():
    name = code_name.get(code, code)
    base_price = df["close"].iloc[0]
    if base_price and float(base_price) > 0:
        cum_ret = (df["close"].astype(float) / float(base_price) - 1) * 100
        fig.add_trace(go.Scatter(
            x=df["trade_date"], y=cum_ret, name=name,
            line=dict(width=1.8),
        ))

# ETF
for code in etf_codes:
    name = code_name.get(code, code)
    etf_df = base_repo.read_sql(
        "SELECT trade_date, close FROM etf_daily "
        "WHERE code = :code AND trade_date BETWEEN :s AND :e ORDER BY trade_date",
        {"code": code, "s": _start, "e": _end},
    )
    if not etf_df.empty:
        etf_df["trade_date"] = pd.to_datetime(etf_df["trade_date"], format="%Y%m%d")
        base_price = float(etf_df["close"].iloc[0])
        if base_price > 0:
            cum_ret = (etf_df["close"].astype(float) / base_price - 1) * 100
            fig.add_trace(go.Scatter(
                x=etf_df["trade_date"], y=cum_ret, name=name,
                line=dict(width=1.8, dash="dash"),
            ))

fig.add_hline(y=0, line_dash="solid", line_color="gray", opacity=0.4)
fig.update_layout(title="累计收益率对比 (%)", yaxis_title="累计收益 (%)",
                  height=450, template="plotly_white", margin=dict(t=40, b=20))
fig.show()

### 3.8 数据归因汇总

汇总每只标的的最新技术指标快照，并给出简要归因判断。

In [77]:
def _boll_position(row):
    close = float(row.get("close", 0) or 0)
    upper = float(row.get("boll_upper", 0) or 0)
    lower = float(row.get("boll_lower", 0) or 0)
    mid = float(row.get("boll_mid", 0) or 0)
    if upper == lower:
        return "—"
    pct = (close - lower) / (upper - lower) if (upper - lower) != 0 else 0.5
    if pct > 0.8:
        return f"上轨附近 ({pct:.0%})"
    elif pct < 0.2:
        return f"下轨附近 ({pct:.0%})"
    else:
        return f"中轨区域 ({pct:.0%})"

def _macd_signal(row):
    dif = float(row.get("macd_dif", 0) or 0)
    dea = float(row.get("macd_dea", 0) or 0)
    if dif > dea and dif > 0:
        return "多头强势"
    elif dif > dea:
        return "金叉向上"
    elif dif < dea and dif < 0:
        return "空头主导"
    else:
        return "死叉向下"

def _rsi_signal(row):
    rsi = float(row.get("rsi_14", 50) or 50)
    if rsi > 70:
        return f"超买 ({rsi:.1f})"
    elif rsi < 30:
        return f"超卖 ({rsi:.1f})"
    else:
        return f"中性 ({rsi:.1f})"

summary_rows = []
for code, df in stock_data.items():
    name = code_name.get(code, code)
    last = df.iloc[-1]

    summary_rows.append({
        "名称": name,
        "代码": code,
        "最新收盘": f"{float(last['close']):.2f}",
        "RSI(14)": _rsi_signal(last),
        "MACD方向": _macd_signal(last),
        "布林带位置": _boll_position(last),
        "MA60偏离": f"{float(last.get('ma60_bias', 0) or 0):.2%}",
        "20日动量": f"{float(last.get('momentum_20', 0) or 0):.2%}",
        "20日波动率": f"{float(last.get('volatility_20', 0) or 0):.2%}",
        "量比": f"{float(last.get('vol_ratio', 0) or 0):.2f}",
    })

df_summary = pd.DataFrame(summary_rows)
print("各标的最新技术指标快照:\n")
display(df_summary)

print("\n归因说明:")
for _, row in df_summary.iterrows():
    signals = []
    if "超买" in row["RSI(14)"]:
        signals.append("RSI 超买区域，短期或有回调压力")
    elif "超卖" in row["RSI(14)"]:
        signals.append("RSI 超卖区域，可能接近反弹")

    if "多头" in row["MACD方向"]:
        signals.append("MACD 多头强势，趋势向上")
    elif "金叉" in row["MACD方向"]:
        signals.append("MACD 金叉，趋势转多")
    elif "空头" in row["MACD方向"]:
        signals.append("MACD 空头主导，趋势偏弱")
    elif "死叉" in row["MACD方向"]:
        signals.append("MACD 死叉，需关注下行风险")

    if "上轨" in row["布林带位置"]:
        signals.append("价格接近布林上轨，波动加大")
    elif "下轨" in row["布林带位置"]:
        signals.append("价格接近布林下轨，存在支撑")

    bias = float(row["MA60偏离"].rstrip("%")) / 100
    if abs(bias) > 0.1:
        direction = "大幅偏离上方" if bias > 0 else "大幅偏离下方"
        signals.append(f"MA60 {direction}，均值回归概率增大")

    vol = float(row["量比"])
    if vol > 2:
        signals.append(f"量比 {vol:.1f}，成交显著放量")
    elif vol < 0.5:
        signals.append(f"量比 {vol:.1f}，成交极度萎缩")

    summary = "；".join(signals) if signals else "各指标表现平稳，无明显信号"
    print(f"  【{row['名称']}】{summary}")

各标的最新技术指标快照:



,名称,代码,最新收盘,RSI(14),MACD方向,布林带位置,MA60偏离,20日动量,20日波动率,量比
0,粤桂股份,000833.SZ,31.72,超买 (73.2),多头强势,上轨附近 (251%),29.56%,26.63%,65.65%,1.18
1,比亚迪,002594.SZ,102.98,中性 (52.4),死叉向下,上轨附近 (125%),4.71%,0.32%,32.78%,1.02
2,卫星化学,002648.SZ,29.60,中性 (61.3),多头强势,上轨附近 (164%),13.73%,9.59%,51.86%,0.99
3,润泽科技,300442.SZ,89.06,中性 (49.8),死叉向下,上轨附近 (96%),1.86%,5.35%,56.66%,0.84
4,潞化科技,600691.SH,3.35,中性 (50.7),金叉向上,上轨附近 (157%),-2.01%,2.13%,51.91%,0.64



归因说明:
  【粤桂股份】RSI 超买区域，短期或有回调压力；MACD 多头强势，趋势向上；价格接近布林上轨，波动加大；MA60 大幅偏离上方，均值回归概率增大
  【比亚迪】MACD 死叉，需关注下行风险；价格接近布林上轨，波动加大
  【卫星化学】MACD 多头强势，趋势向上；价格接近布林上轨，波动加大；MA60 大幅偏离上方，均值回归概率增大
  【润泽科技】MACD 死叉，需关注下行风险；价格接近布林上轨，波动加大
  【潞化科技】MACD 金叉，趋势转多；价格接近布林上轨，波动加大
